<a href="https://colab.research.google.com/github/nihaal237/SLMFIX-Leveraging-Small-LLMs-for-Error-Fixing-with-Reinforcement-Learning/blob/main/SLMFix_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Update bitsandbytes to the latest version required for 4-bit
!pip install -U bitsandbytes>=0.46.1

# 2. Update transformers and accelerate to match
!pip install -U transformers accelerate huggingface_hub

print("✅ Libraries updated. ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 31.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
✅ Libraries updated. 


In [ ]:
from google.colab import auth, drive #connecting to the shared drive folder
import os

# 1. Login via Browser
print("🔑 Authenticating...")
auth.authenticate_user()

# 2. Connect the Drive (with "Already Mounted" fix)
print("📂 Checking Drive connection...")
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive mounted successfully.")
except Exception as e:
    if "already contain files" in str(e):
        print("ℹ️ Drive was already mounted. Skipping...")
    else:
        print(f"❌ Error mounting drive: {e}")


🔑 Authenticating...
📂 Checking Drive connection...
Mounted at /content/drive
✅ Drive mounted successfully.


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/SLMFix_Project"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig  #load model directly from drive

# path Variables

MODEL_DIR = os.path.join(PROJECT_ROOT, "qwen_coder_7b_4bit")


print("⚡ Loading Qwen Coder model from Shared Drive (Fast Mode)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        device_map="auto",
        local_files_only=True,
        low_cpu_mem_usage=True
    )

print("🚀 Qwen is ready!")

⚡ Loading Qwen Coder model from Shared Drive (Fast Mode)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

🚀 Qwen is ready!


In [ ]:
def proposer(instruction, task="sql"):
    system_msg = f"You are a coding expert. Provide ONLY the {task} code, no explanations." #test model on sql
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": instruction}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the code
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1)
    prediction = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

    return prediction.strip()

# TEST IT:
print("🧪 Test SQL:", proposer("Show names of all students in 'Computer Science'"))

In [ ]:
import json #loading Bash datasets from drive

with open("/content/drive/MyDrive/SLMFix_Project/bash_datasets/train.json") as f:
    train_bdata = json.load(f)

with open("/content/drive/MyDrive/SLMFix_Project/bash_datasets/dev.json") as f:
    dev_bdata = json.load(f)

with open("/content/drive/MyDrive/SLMFix_Project/bash_datasets/test.json") as f:
    test_bdata = json.load(f)

print("Train samples:", len(train_bdata))
print("Dev samples:", len(dev_bdata))
print("Test samples:", len(test_bdata))
print("\nSample:", train_bdata[0])

Train samples: 8090
Dev samples: 609
Test samples: 606

Sample: {'nl': "Do a dry run of renaming file extension '.andnav' to '.tile' for all files/directories under current directory tree", 'bash': 'find . -name "*.andnav" | rename -vn "s/\\.andnav$/.tile/"'}


In [ ]:
import os #loading SQL datasets from drive
import json

with open("/content/drive/MyDrive/SLMFix_Project/spider_data/spider_data/train_spider.json") as f:
        train_sdata=json.load(f)

with open("/content/drive/MyDrive/SLMFix_Project/spider_data/spider_data/test.json") as f:
        test_sdata=json.load(f)

with open("/content/drive/MyDrive/SLMFix_Project/spider_data/spider_data/dev.json") as f:
        dev_sdata=json.load(f)


print(f"Training samples: {len(train_sdata)}")
print(f"Dev samples: {len(dev_sdata)}")
print(f"Test samples: {len(test_sdata)}")
print("\nSample: ",train_sdata[0]) #db_id in each sample indicates the name of the corresponding database in database sub-folder in spider_data


Training samples: 7000
Dev samples: 1034
Test samples: 2147

Sample:  {'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question': 'How many heads of the departments are older than 56 ?', 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?'], 'sql': {'from': {'table_units': [['table_unit', 1]], 'conds': []}, 'select': [False, [[3, [0, [0, 0, False], None]]]], 'where': [[False, 3, [0, [0, 10, False], None], 56.0, None]], 'groupBy': [], 'having': [], 'orderBy': [], 'limit': None, 'intersect': None, 'union': None, 'except': None}}


In [ ]:
import sqlite3

def get_db_schema(db_id): #return database tables associated with the sql query


    db_path = f"/content/drive/MyDrive/SLMFix_Project/spider_data/spider_data/database/{db_id}/{db_id}.sqlite"

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Get all table names
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [t[0] for t in cursor.fetchall()]

        schema_text = ""
        for table in tables:
            cursor.execute(f"PRAGMA table_info({table});")
            columns = [col[1] for col in cursor.fetchall()]
            schema_text += f"Table: {table}, columns: {', '.join(columns)} | "

        conn.close()

        return schema_text
    except Exception as e:
        return f"Error reading schema: {e}"

print("Schema for Sample 0:", get_db_schema(train_sdata[0]['db_id']))

Schema for Sample 0: Table: department, columns: Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees | Table: head, columns: head_ID, name, born_state, age | Table: management, columns: department_ID, head_ID, temporary_acting | 


In [ ]:
def generate_sql(question, schema):
    """
    Generate SQL from the 7B model using Qwen chat template.
    Raw prompt strings produce empty output — chat format is required.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert SQL assistant. "
                "Given a database schema and a question, write ONLY the SQL query. "
                "No explanation, no markdown, no extra text."
            )
        },
        {
            "role": "user",
            "content": f"Database Schema:\n{schema}\n\nQuestion: {question}\n\nWrite the SQL query:"
        }
    ]

    # apply the chat template the model was trained with
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,               # greedy — deterministic output
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # decode only newly generated tokens, not the prompt
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return generated

# quick test
test_sample = dev_sdata[0]
ai_sql      = generate_sql(test_sample["question"], get_db_schema(test_sample["db_id"]))
print(f"Question  : {test_sample['question']}")
print(f"Generated : {ai_sql}")
print(f"Gold SQL  : {test_sample['query']}")

NameError: name 'tokenizer' is not defined

In [ ]:
import re
import sqlglot
from sqlglot import exp

def clean_sql(raw_sql):
    """Strip markdown fences and trailing semicolons from model output."""
    clean = re.sub(r'```sql|```', '', raw_sql)
    clean = clean.strip().rstrip(';').strip()
    return clean

def compute_ast_similarity(predicted_sql, ground_truth_sql):
    """Jaccard similarity between AST node type sets — used as semantic reward."""
    try:
        pred_nodes = {type(n).__name__ for n in sqlglot.parse_one(clean_sql(predicted_sql),  dialect="sqlite").walk()}
        gold_nodes = {type(n).__name__ for n in sqlglot.parse_one(clean_sql(ground_truth_sql), dialect="sqlite").walk()}
        intersection = pred_nodes & gold_nodes
        union        = pred_nodes | gold_nodes
        return len(intersection) / len(union) if union else 0.0
    except Exception:
        return 0.0  # unparseable gets zero

def validate_sql_combined(predicted_sql, db_id, schema=None):
    """
    Stage 1 — SQLGlot static check  → sets rl_reward (used in PPO training)
    Stage 2 — SQLite execution check → sets fully_valid (used in evaluation)
    """
    sql    = clean_sql(predicted_sql)
    result = {
        "static_valid":   False,
        "static_error":   None,
        "runtime_valid":  False,
        "runtime_error":  None,
        "runtime_output": None,
        "rl_reward":      0.0,
        "fully_valid":    False,
    }

    # stage 1: static syntax check — safe, fast, no DB needed
    try:
        sqlglot.parse_one(sql, dialect="sqlite")
        result["static_valid"] = True
        result["rl_reward"]    = 1.0
    except sqlglot.errors.ParseError as e:
        result["static_error"] = str(e)
        return result  # broken — no point running stage 2

    # optional schema check — catches hallucinated table/column names
    if schema is not None:
        try:
            parsed        = sqlglot.parse_one(sql, dialect="sqlite")
            ref_tables    = {t.name.lower() for t in parsed.find_all(exp.Table)}
            ref_cols      = {c.name.lower() for c in parsed.find_all(exp.Column)}
            schema_tables = {t.lower() for t in schema.keys()}
            schema_cols   = {c.lower() for cols in schema.values() for c in cols}

            if ref_tables - schema_tables:
                result["static_valid"] = False
                result["static_error"] = f"Unknown tables: {ref_tables - schema_tables}"
                result["rl_reward"]    = 0.0
                return result

            if ref_cols - schema_cols:
                result["static_valid"] = False
                result["static_error"] = f"Unknown columns: {ref_cols - schema_cols}"
                result["rl_reward"]    = 0.0
                return result
        except Exception as e:
            result["static_error"] = f"Schema check failed: {e}"

    # stage 2: runtime execution — only for final evaluation, never for RL
    db_path = os.path.join(
        PROJECT_ROOT,
        f"spider_data/spider_data/database/{db_id}/{db_id}.sqlite"
    )
    try:
        conn   = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows   = cursor.fetchall()
        conn.close()
        result["runtime_valid"]  = True
        result["runtime_output"] = rows
        result["fully_valid"]    = True
    except Exception as e:
        result["runtime_error"] = str(e)

    return result

def compute_batch_pass_rate(predicted_sqls, db_ids):
    """Fraction of queries in current batch passing static validation."""
    results = [validate_sql_combined(sql, db_id) for sql, db_id in zip(predicted_sqls, db_ids)]
    return sum(r["rl_reward"] for r in results) / len(results)

def compute_reward(predicted_sql, ground_truth_sql, db_id, batch_pass_rate):
    """
    Adaptive reward from the paper:
    early training  → static score dominates (fix syntax first)
    later training  → semantic score dominates (improve correctness)
    """
    static_score   = validate_sql_combined(predicted_sql, db_id)["rl_reward"]
    semantic_score = compute_ast_similarity(predicted_sql, ground_truth_sql)
    pr             = batch_pass_rate
    return (1 - pr) * static_score + pr * semantic_score

def print_validation_report(result):
    print("=" * 50)
    print(f"  Static  valid : {result['static_valid']}")
    if result["static_error"]:
        print(f"  Static  error : {result['static_error']}")
    print(f"  Runtime valid : {result['runtime_valid']}")
    if result["runtime_error"]:
        print(f"  Runtime error : {result['runtime_error']}")
    if result["runtime_output"] is not None:
        print(f"  Rows returned : {len(result['runtime_output'])}")
        print(f"  Sample output : {result['runtime_output'][:3]}")
    print(f"  RL reward     : {result['rl_reward']}")
    print(f"  Fully valid   : {result['fully_valid']}")
    print("=" * 50)

# quick test
result = validate_sql_combined(ai_sql, test_sample['db_id'])
print_validation_report(result)

NameError: name 'ai_sql' is not defined

In [ ]:
from tqdm import tqdm
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "generated_data/training_triples.json")
os.makedirs(os.path.join(PROJECT_ROOT, "generated_data"), exist_ok=True)

def generate_training_triples(spider_data, output_path, max_samples=None, similarity_threshold=0.8):
    if max_samples:
        spider_data = spider_data[:max_samples]

    # ── resume logic: load existing progress from Drive if it exists ──────────
    if os.path.exists(output_path):
        with open(output_path, "r") as f:
            triples = json.load(f)
        # figure out how many spider samples were already processed
        # we track this by saving a separate progress file
        progress_path = output_path.replace(".json", "_progress.json")
        if os.path.exists(progress_path):
            with open(progress_path, "r") as f:
                progress = json.load(f)
            start_index = progress["last_processed_index"] + 1
        else:
            start_index = 0
        print(f"⚡ Resuming from sample {start_index} ({len(triples)} triples already saved)")
    else:
        triples     = []
        start_index = 0
        print(f"🆕 Starting fresh run")

    # only process samples we haven't seen yet
    remaining_data = spider_data[start_index:]

    type_a_count = 0
    type_b_count = 0
    type_c_count = 0
    total        = len(spider_data)

    print(f"Generating SQL for {len(remaining_data)} remaining samples...")

    for i, sample in enumerate(tqdm(remaining_data)):
        actual_index = start_index + i   # real index in the full dataset
        question     = sample["question"]
        gold_sql     = sample["query"]
        db_id        = sample["db_id"]
        schema       = get_db_schema(db_id)

        predicted_sql = generate_sql(question, schema)
        validation    = validate_sql_combined(predicted_sql, db_id)
        broken        = False
        error_msg     = None
        failure_type  = None

        # type A: static syntax failure
        if not validation["static_valid"]:
            broken       = True
            error_msg    = validation["static_error"]
            failure_type = "static_failure"
            type_a_count += 1

        # type B: runtime failure
        elif not validation["runtime_valid"]:
            broken       = True
            error_msg    = validation["runtime_error"]
            failure_type = "runtime_failure"
            type_b_count += 1

        # type C: semantic mismatch
        else:
            similarity = compute_ast_similarity(predicted_sql, gold_sql)
            if similarity < similarity_threshold:
                broken       = True
                error_msg    = f"Low semantic similarity: {similarity:.3f}"
                failure_type = "semantic_mismatch"
                type_c_count += 1

        if broken:
            triples.append({
                "question":     question,
                "db_id":        db_id,
                "schema":       schema,
                "broken_sql":   predicted_sql,
                "error_msg":    error_msg,
                "failure_type": failure_type,
                "gold_sql":     gold_sql,
                "rl_reward":    0.0,
            })

        # save every 100 samples — checkpoint both triples and progress index
        if (actual_index + 1) % 100 == 0:
            with open(output_path, "w") as f:
                json.dump(triples, f, indent=2)
            # save progress index so we know where to resume from
            with open(output_path.replace(".json", "_progress.json"), "w") as f:
                json.dump({"last_processed_index": actual_index}, f)
            print(
                f"  [{actual_index+1}/{total}] "
                f"A:{type_a_count} B:{type_b_count} C:{type_c_count} "
                f"total broken:{len(triples)}"
            )

    # final save
    with open(output_path, "w") as f:
        json.dump(triples, f, indent=2)
    with open(output_path.replace(".json", "_progress.json"), "w") as f:
        json.dump({"last_processed_index": total - 1}, f)

    print(f"\n✅ Done!")
    print(f"   Total processed        : {total}")
    print(f"   Type A static failures : {type_a_count}")
    print(f"   Type B runtime failures: {type_b_count}")
    print(f"   Type C semantic mismatches: {type_c_count}")
    print(f"   Total broken collected : {len(triples)}")
    print(f"   7B clean pass rate     : {((total - len(triples)) / total) * 100:.1f}%")
    print(f"   Saved to               : {output_path}")
    return triples

In [ ]:
# inspect what the model generates for the first 10 training samples
for i, sample in enumerate(train_sdata[:10]):
    question  = sample["question"]
    gold_sql  = sample["query"]
    db_id     = sample["db_id"]
    schema    = get_db_schema(db_id)

    predicted = generate_sql(question, schema)
    validation = validate_sql_combined(predicted, db_id)

    print(f"── Sample {i+1} ──────────────────────────────────")
    print(f"  Question   : {question}")
    print(f"  Generated  : {predicted}")
    print(f"  Gold SQL   : {gold_sql}")
    print(f"  Static OK  : {validation['static_valid']}")
    print(f"  Runtime OK : {validation['runtime_valid']}")
    print(f"  RL Reward  : {validation['rl_reward']}")
    if validation['static_error']:
        print(f"  Error      : {validation['static_error']}")
    print()

In [ ]:
# ── paste this into a new cell and run it before starting the full run ────────
# keeps the Colab session alive by clicking the page every 60 seconds

from IPython.display import display, Javascript

display(Javascript('''
function clickConnect() {
    console.log("Keeping session alive...");
    document.querySelector("#top-toolbar > colab-connect-button")
        ?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(clickConnect, 60000);  // every 60 seconds
'''))

print("✅ Session keep-alive started")

<IPython.core.display.Javascript object>

✅ Session keep-alive started


In [ ]:
# ── full run — safe to restart anytime ───────────────────────────────────────
all_triples = generate_training_triples(
    train_sdata,
    OUTPUT_PATH,
    max_samples=None,
    similarity_threshold=0.8
)

In [ ]:
import json
import torch
import sqlite3
import re
import os
from tqdm import tqdm

# THE FIXER FUNCTION ---
def fix_sql(question, schema, original_sql, error_message):
    """
    Stage 4: The SLM-Fixer.
    Provides the error feedback to the model to get a corrected query.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert SQL assistant. Use the provided error message "
                "to fix the SQL query. Return ONLY the corrected SQL query. "
                "No explanation, no markdown."
            )
        },
        {
            "role": "user",
            "content": (
                f"Database Schema:\n{schema}\n\n"
                f"Question: {question}\n\n"
                f"Your previous SQL: {original_sql}\n\n"
                f"Error from Database: {error_message}\n\n"
                "Fixed SQL query:"
            )
        }
    ]

    # Apply the chat template (assumes 'tokenizer' and 'model' are already in memory)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return generated


# RESUME-FRIENDLY RECOVERY LOOP

INPUT_PATH = "/content/drive/MyDrive/SLMFix_Project/generated_data/training_triples.json"
OUTPUT_PATH = "/content/drive/MyDrive/SLMFix_Project/generated_data/recovery_results.json"

# 1. Load the broken samples
with open(INPUT_PATH, "r") as f:
    broken_triples = json.load(f)

# 2. Check for existing progress
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        results = json.load(f)
    start_index = len(results)
    print(f"⚡ Resuming from sample {start_index} ({len(results)} already processed in a previous run)")
else:
    results = []
    start_index = 0
    print("🆕 Starting fresh run for recovery")

# 3. Filter only the remaining samples
remaining_triples = broken_triples[start_index:]

# Count how many have been recovered from previous runs
recovered_count = sum(1 for r in results if r.get("status") == "fixed")

print(f"Attempting to fix {len(remaining_triples)} remaining samples...")

for i, entry in enumerate(tqdm(remaining_triples)):
    actual_index = start_index + i  # Real index out of 1634

    # Stage 4: Call the Fixer
    fixed_sql = fix_sql(
        entry["question"],
        entry["schema"],
        entry["broken_sql"],
        entry["error_msg"]
    )

    # Stage 3 (Again): Validate the fix
    validation = validate_sql_combined(fixed_sql, entry["db_id"])

    if validation["fully_valid"]:
        recovered_count += 1
        entry["status"] = "fixed"
    else:
        entry["status"] = "still_broken"
        entry["final_error"] = validation.get("runtime_error") or validation.get("static_error")

    entry["fixed_sql"] = fixed_sql
    results.append(entry)

    # Checkpoint saving every 50 samples so you don't lose data on Google Colab Timeouts
    if (actual_index + 1) % 50 == 0:
        with open(OUTPUT_PATH, "w") as f:
            json.dump(results, f, indent=2)


# 4. Final Save
with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

# Calculate New Stats (Based on your 7000 sample run)
initial_correct = 7000 - 1634
final_total_correct = initial_correct + recovered_count
new_accuracy = (final_total_correct / 7000) * 100

print(f"\n✅ Recovery Checkpoint Complete!")
print(f"Total Fixed: {recovered_count} / {len(results)}")
print(f"New Overall Accuracy: {new_accuracy:.1f}%")

Bash

In [ ]:
import re
import json
import os
from tqdm import tqdm
!pip install bashlex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
def clean_bash(raw_bash):
    clean = re.sub(r'```bash|```', '', raw_bash)
    clean = clean.strip()
    return clean


In [ ]:
def validate_bash_static(command):
    """Stage 1 — bashlex static parse, feeds rl_reward. Never executes the command."""
    import bashlex
    try:
        bashlex.parse(command)
        return True, None
    except Exception as e:
        return False, str(e)

In [ ]:
COMMON_UTILS = {
    "find", "grep", "sed", "awk", "ls", "cat", "echo", "cp", "mv", "rm",
    "mkdir", "chmod", "chown", "tar", "gzip", "gunzip", "sort", "uniq",
    "wc", "head", "tail", "cut", "tr", "xargs", "tee", "diff", "touch",
    "pwd", "cd", "export", "read", "printf", "test", "curl", "wget"
}

def validate_bash_heuristic(command):
    """
    Stage 2 — heuristic utility/flag whitelist check.
    Used for triple construction only — commands are never executed.
    Excludes unvalidatable constructs like here-docs and process substitutions.
    """
    # exclude unvalidatable constructs as described in Section 3.2
    if '<<' in command:                     # here-docs
        return False, "Contains here-doc (<<)"
    if '<(' in command or '>(' in command:  # process substitution
        return False, "Contains process substitution"

    # check that the first token is a known utility
    first_token = command.strip().split()[0] if command.strip() else ""
    if first_token not in COMMON_UTILS:
        return False, f"Unknown or unsupported utility: {first_token}"

    return True, None

In [ ]:
def compute_bash_similarity(predicted_bash, gold_bash):
    """
    Token-level F1 between generated and gold command.
    Used as the semantic score in the PPO reward function.
    """
    pred_tokens = set(predicted_bash.strip().split())
    gold_tokens = set(gold_bash.strip().split())

    if not pred_tokens or not gold_tokens:
        return 0.0

    overlap   = pred_tokens & gold_tokens
    precision = len(overlap) / len(pred_tokens)
    recall    = len(overlap) / len(gold_tokens)

    if precision + recall == 0:
        return 0.0

    f1 = 2 * precision * recall / (precision + recall)
    return f1

In [ ]:
def validate_bash_combined(predicted_bash):
    """
    Stage 1 — bashlex static parse  → sets rl_reward (used in PPO training)
    Stage 2 — heuristic whitelist   → used for triple construction only
    """
    command = clean_bash(predicted_bash)

    result = {
        "static_valid":    False,
        "static_error":    None,
        "heuristic_valid": False,
        "heuristic_error": None,
        "rl_reward":       0.0,
        "fully_valid":     False,
    }

    # stage 1: bashlex static parse — safe, no execution
    static_ok, static_err = validate_bash_static(command)
    if not static_ok:
        result["static_error"] = static_err
        return result

    result["static_valid"] = True
    result["rl_reward"]    = 1.0   # reward for PPO training

    # stage 2: heuristic check — triple construction only
    heuristic_ok, heuristic_err = validate_bash_heuristic(command)
    result["heuristic_valid"] = heuristic_ok
    result["heuristic_error"] = heuristic_err
    result["fully_valid"]     = heuristic_ok

    return result

In [ ]:
def compute_bash_reward(predicted_bash, gold_bash, batch_pass_rate):
    """
    PPO reward = (1-pr) × static_score + pr × semantic_score
    Adaptive weighting: early training focuses on syntax, later on semantics.
    """
    static_score   = validate_bash_combined(predicted_bash)["rl_reward"]
    semantic_score = compute_bash_similarity(predicted_bash, gold_bash)
    pr             = batch_pass_rate
    return (1 - pr) * static_score + pr * semantic_score


In [ ]:
def generate_bash(invocation):
    """Generate a Bash command from the 7B model using chat template format."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Bash programmer. "
                "Given a task description, write ONLY the bash command. "
                "No explanation, no markdown, no extra text."
            )
        },
        {
            "role": "user",
            "content": f"Task: {invocation}\n\nWrite the bash command:"
        }
    ]

    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,          # bash commands are short
            do_sample=False,             # greedy decoding
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return generated

In [ ]:
BASH_OUTPUT_PATH    = os.path.join(PROJECT_ROOT, "generated_data/bash_training_triples.json")
BASH_PROGRESS_PATH  = BASH_OUTPUT_PATH.replace(".json", "_progress.json")

def generate_bash_triples(bash_data, output_path, max_samples=None, similarity_threshold=0.8):
    """
    Run NL2Bash training samples through the 7B proposer.
    Collect broken or semantically mismatched outputs as training triples.
    Each triple: invocation, gold_cmd, broken_cmd, error_msg, failure_type, rl_reward.
    """
    if max_samples:
        bash_data = bash_data[:max_samples]

    # resume logic — load existing progress if present
    progress_path = output_path.replace(".json", "_progress.json")
    if os.path.exists(output_path):
        with open(output_path, "r") as f:
            triples = json.load(f)
        if os.path.exists(progress_path):
            with open(progress_path, "r") as f:
                start_index = json.load(f)["last_processed_index"] + 1
        else:
            start_index = 0
        print(f"⚡ Resuming Bash from sample {start_index} ({len(triples)} triples saved)")
    else:
        triples     = []
        start_index = 0
        print("🆕 Starting fresh Bash run")

    remaining  = bash_data[start_index:]
    total      = len(bash_data)
    type_a = type_b = type_c = 0

    print(f"Generating Bash for {len(remaining)} remaining samples...")

    for i, sample in enumerate(tqdm(remaining)):
        actual_index = start_index + i
        invocation   = sample.get("invocation", sample.get("nl", ""))  # field name varies
        gold_cmd     = sample.get("cmd", sample.get("bash", ""))

        predicted_bash = generate_bash(invocation)
        validation     = validate_bash_combined(predicted_bash)
        broken         = False
        error_msg      = None
        failure_type   = None

        # type A: bashlex rejects — static failure
        if not validation["static_valid"]:
            broken       = True
            error_msg    = validation["static_error"]
            failure_type = "static_failure"
            type_a      += 1

        # type B: bashlex passes but heuristic check fails
        elif not validation["heuristic_valid"]:
            broken       = True
            error_msg    = validation["heuristic_error"]
            failure_type = "runtime_failure"
            type_b      += 1

        # type C: both pass but token F1 below threshold
        else:
            similarity = compute_bash_similarity(predicted_bash, gold_cmd)
            if similarity < similarity_threshold:
                broken       = True
                error_msg    = f"Low token F1 similarity: {similarity:.3f}"
                failure_type = "semantic_mismatch"
                type_c      += 1

        if broken:
            triples.append({
                "invocation":   invocation,
                "broken_cmd":   predicted_bash,
                "error_msg":    error_msg,
                "failure_type": failure_type,
                "gold_cmd":     gold_cmd,
                "rl_reward":    0.0,
            })

        # checkpoint every 100 samples — protects against Colab disconnects
        if (actual_index + 1) % 100 == 0:
            with open(output_path, "w") as f:
                json.dump(triples, f, indent=2)
            with open(progress_path, "w") as f:
                json.dump({"last_processed_index": actual_index}, f)
            print(f"  [{actual_index+1}/{total}] A:{type_a} B:{type_b} C:{type_c} total:{len(triples)}")

    # final save
    with open(output_path, "w") as f:
        json.dump(triples, f, indent=2)
    with open(progress_path, "w") as f:
        json.dump({"last_processed_index": total - 1}, f)

    print(f"\n✅ Bash Done!")
    print(f"   Total processed        : {total}")
    print(f"   Type A static failures : {type_a}")
    print(f"   Type B heuristic fails : {type_b}")
    print(f"   Type C semantic mismatches: {type_c}")
    print(f"   Total triples collected: {len(triples)}")
    print(f"   Saved to               : {output_path}")
    return triples

In [ ]:
print("--- BASH TEST RUN (10 samples) ---")
bash_test = generate_bash_triples(train_bdata, BASH_OUTPUT_PATH, max_samples=10)


--- BASH TEST RUN (10 samples) ---
🆕 Starting fresh Bash run
Generating Bash for 10 remaining samples...


 50%|█████     | 5/10 [00:10<00:10,  2.01s/it]

In [ ]:
all_bash_triples = generate_bash_triples(train_bdata, BASH_OUTPUT_PATH, max_samples=None)

⚡ Resuming Bash from sample 10 (10 triples saved)
Generating Bash for 8080 remaining samples...


  1%|          | 90/8080 [03:39<5:50:53,  2.64s/it]

  [100/8090] A:0 B:45 C:44 total:99


  2%|▏         | 190/8080 [07:15<5:11:08,  2.37s/it]

  [200/8090] A:1 B:88 C:97 total:196


...

  5%|▍         | 390/8080 [13:36<7:05:25,  3.32s/it]

  [400/8090] A:2 B:138 C:234 total:384


  6%|▌         | 490/8080 [17:23<3:58:30,  1.89s/it]

  [500/8090] A:3 B:180 C:287 total:480


  7%|▋         | 588/8080 [20:35<4:14:03,  2.03s/it]